# Laboratorio 1 · Bitácora

**Nombre:BRAYAN ANYELO RODRIGUEZ LANCHEROS**   
**Usuario de GitHub:Anyelorodriguez** 
**Fecha:28/08/2026**  

---

> Los enunciados están en la guía del laboratorio. Aquí solo van tus
> predicciones, tus resultados y tus explicaciones.

> **La regla que hace que esto sirva de algo:** la predicción se escribe
> *antes* de ejecutar. Si la rellenas después ya sabiendo el resultado, el
> ejercicio no mide nada y tú no aprendes nada. Nadie va a comprobarlo:
> es un trato contigo mismo.


In [1]:
from rlrs.dp import greedy_policy, q_value, value_iteration
from rlrs.envs import ACTION_NAMES, GridWorld
from rlrs.evaluation import compare, evaluate
from rlrs.policies import GreedyTabularPolicy, RandomPolicy

# Este cuaderno es una bitácora: no define algoritmos, los usa.
# Si necesitas escribir una función que valga la pena conservar,
# va en src/rlrs/, no aquí.


## Mi variante

Ejecuta `uv run python scripts/variante.py` y anota lo que te tocó.


In [2]:
RUIDO  = 0.2   # <- rellena
COSTE  = -0.02   # <- rellena
GAMMA  = 0.9

mi_env = GridWorld(noise=RUIDO, step_reward=COSTE)
mi_env.noise, mi_env.step_reward


(0.2, -0.02)

---
## Ejercicio 1 · El respaldo a mano


**Antes de ejecutar.** ¿Cuál de las cuatro acciones crees que gana en (0,3), y por qué?

_Tu predicción:_ Creo que la acción que gana en (0,3) será la que permita avanzar hacia una trayectoria con mayor valor esperado, teniendo en cuenta el ruido del entorno y las recompensas futuras en este caso a la derecha


In [ ]:
# tu código
from rlrs.dp import q_value, value_iteration
from rlrs.envs import ACTION_NAMES, GridWorld

env = GridWorld()                      # el de la guía, no tu variante
valores, politica, barridos = value_iteration(env, gamma=0.9)

for a in range(env.n_actions):
    print(f'{ACTION_NAMES[a]:<10} {q_value(env, valores, (0, 3), a, 0.9):+.6f}')

arriba     +0.800017
derecha    +0.928402
abajo      +0.554337
izquierda  +0.636940


**Explica.** ¿Coincidió? ¿Por qué gana esa y no las otras?

_Tu respuesta:_ Sí, la acción ganadora fue derecha, con un valor \(Q(0,3,\text{derecha})=0.928402\). Esta acción tiene el mayor valor de las cuatro alternativas. Esto significa que, considerando las transiciones del entorno, las recompensas futuras y un factor de descuento \(\gamma=0.9\), moverse hacia la derecha produce el mayor retorno esperado desde el estado (0,3).


---
## Ejercicio 2 · Tu variante, medida


In [4]:
from rlrs.evaluation import compare
from rlrs.policies import GreedyTabularPolicy, RandomPolicy

mi_env = GridWorld(noise=RUIDO, step_reward=COSTE)     # los tuyos
valores, politica, barridos = value_iteration(mi_env, gamma=0.9)
print(f'{barridos} barridos · V(3,0) = {valores[mi_env.state_index((3, 0))]:+.4f}')
print(mi_env.render_values(valores, politica))

for r in compare(mi_env,
                 [GreedyTabularPolicy(politica, name='optima'),
                  RandomPolicy(mi_env.n_actions, seed=0)],
                 episodes=300, base_seed=0):
    print(' ', r)

35 barridos · V(3,0) = +0.3178
+0.55>  +0.66>  +0.78>  +0.94>   +1    
+0.46^    ###   +0.66^  +0.61^   -1    
+0.38^  +0.44>  +0.54^    ###   +0.23v 
+0.32^  +0.37^  +0.44^  +0.36<  +0.28< 
  optima         retorno +0.798 [+0.766, +0.831]  exito 98.0%  pasos   9.1
  aleatoria      retorno -1.400 [-1.511, -1.289]  exito 25.7%  pasos  56.8


La iteración de valores necesitó 35 barridos para converger en mi variante, con ruido de 0.2, coste por paso de −0.02 y gamma=0.9. El valor del estado (3,0) fue V(3,0)=0.3178.

Al evaluar la política óptima se obtuvo un retorno medio de +0.798, con un intervalo de [+0.766,+0.831], una tasa de éxito del 98.0 % y un promedio de 9.1 pasos. Esto indica que la política aprendida consigue llegar al objetivo de forma muy consistente y con trayectorias relativamente cortas.

En comparación, la política aleatoria obtuvo un retorno de −1.400, una tasa de éxito de solo 25.7 % y necesitó 56.8 pasos en promedio. Esto muestra que la política obtenida mediante value iteration aprovecha la estructura del entorno y selecciona acciones mucho mejores que una política aleatoria.

**Anota.** Barridos, V(3,0), retorno con su intervalo, tasa de éxito y pasos medios.

_Tus números:_ 35 barridos · V(3,0) = +0.3178
+0.55>  +0.66>  +0.78>  +0.94>   +1    
+0.46^    ###   +0.66^  +0.61^   -1    
+0.38^  +0.44>  +0.54^    ###   +0.23v 
+0.32^  +0.37^  +0.44^  +0.36<  +0.28< 
  optima         retorno +0.798 [+0.766, +0.831]  exito 98.0%  pasos   9.1
  aleatoria      retorno -1.400 [-1.511, -1.289]  exito 25.7%  pasos  56.8


---
## Ejercicio 3 · Subir gamma


**Antes de ejecutar.** Al pasar de 0,9 a 0,99: ¿qué le pasa al número de barridos? ¿Y a la política?

_Tu predicción:_ Al aumentar gamma de 0.9 a 0.99, espero que aumente el número de barridos necesarios para converger, porque las recompensas futuras tendrán mayor peso. Espero que la política cambie poco.


In [5]:
for g in (0.5, 0.9, 0.99):
    v, p, n = value_iteration(mi_env, gamma=g)
    print(f'\ngamma = {g}   {n} barridos   V(3,0) = {v[mi_env.state_index((3, 0))]:+.4f}')
    print(mi_env.render_values(v, p))



gamma = 0.5   20 barridos   V(3,0) = -0.0327
+0.03>  +0.13>  +0.34>  +0.85>   +1    
-0.01^    ###   +0.14^  +0.23^   -1    
-0.02^  -0.01>  +0.04^    ###   -0.04v 
-0.03^  -0.02>  -0.01^  -0.03<  -0.03< 

gamma = 0.9   35 barridos   V(3,0) = +0.3178
+0.55>  +0.66>  +0.78>  +0.94>   +1    
+0.46^    ###   +0.66^  +0.61^   -1    
+0.38^  +0.44>  +0.54^    ###   +0.23v 
+0.32^  +0.37^  +0.44^  +0.36<  +0.28< 

gamma = 0.99   52 barridos   V(3,0) = +0.7605
+0.86>  +0.90>  +0.94>  +0.98>   +1    
+0.82^    ###   +0.90^  +0.87<   -1    
+0.79^  +0.82>  +0.86^    ###   +0.71v 
+0.76^  +0.78^  +0.81^  +0.78<  +0.74< 


**Explica.** ¿Qué se movió mucho y qué se movió poco? ¿Por qué?

_Tu respuesta:_ Al aumentar gamma de 0.9 a 0.99, el número de barridos aumentó de 35 a 52, por lo que mi predicción sobre una convergencia más lenta se cumplió. Esto ocurre porque un gamma más alto da mayor importancia a las recompensas futuras, haciendo que la información sobre el valor de los estados se propague durante más iteraciones.

El valor de V(3,0) también aumentó considerablemente, pasando de +0.3178 con gamma=0.9 a +0.7605 con gamma=0.99 Sin embargo, la política cambió mucho menos que los valores numéricos. Esto muestra que los valores pueden cambiar significativamente sin que necesariamente cambie la acción óptima en todos los estados.


---
## Ejercicio 4 · Quitar el ruido


**Antes de ejecutar.** Con ruido 0, ¿cambia la política óptima respecto a la tuya? ¿En qué casillas?

_Tu predicción:_ Espero que al eliminar el ruido cambie la política óptima en algunas casillas, especialmente en aquellas cercanas a las zonas de peligro. al tener menos ruido no tiene miedo de estar cerca de la zona de peligro y encuentra un camino mas directo.


In [6]:
for ruido in (0.0, 0.2, 0.6):
    e = GridWorld(noise=ruido)
    v, p, n = value_iteration(e, gamma=0.9)
    print(f'\nruido = {ruido}   {n} barridos')
    print(e.render_values(v, p))


ruido = 0.0   9 barridos
+0.62>  +0.73>  +0.86>  +1.00>   +1    
+0.52^    ###   +0.73^  +0.86^   -1    
+0.43^  +0.52>  +0.62^    ###   +0.27v 
+0.34^  +0.43^  +0.52^  +0.43<  +0.34< 

ruido = 0.2   35 barridos
+0.48>  +0.61>  +0.75>  +0.93>   +1    
+0.37^    ###   +0.61^  +0.59^   -1    
+0.28^  +0.36>  +0.47^    ###   +0.10v 
+0.21^  +0.27^  +0.35^  +0.26<  +0.17< 

ruido = 0.6   90 barridos
+0.02^  +0.17>  +0.33>  +0.62>   +1    
-0.08^    ###   +0.21^  +0.28<   -1    
-0.13^  -0.11>  +0.01^    ###   -0.28v 
-0.18^  -0.16>  -0.13<  -0.19<  -0.25v 


**Explica.** ¿Acertaste? Si te sorprendió, di exactamente qué esperabas y qué pasó.

_Tu respuesta:_ Al quitar el ruido, las flechas del camino apenas cambian porque la mejor ruta sigue siendo clara con o sin fallos. Lo que sí cambia es que las casillas ganan más puntos —como la esquina inferior izquierda, que sube de +0.10 a +0.27— porque al no haber riesgo de dar un paso en falso, el premio está asegurado. Además, el algoritmo resuelve el mapa mucho más rápido (baja de 35 a 9 pasos) porque no tiene que calcular probabilidades y todo es directo.


---
## Ejercicio 5 · Encarecer el paso


**Antes de ejecutar.** Con coste por paso −2, ¿qué hará el agente?

_Tu predicción:_ Con un coste por paso de −2, el agente tendrá un incentivo muy fuerte para reducir el número de pasos. Por tanto, espero que prefiera trayectorias muy cortas y que evite permanecer demasiado tiempo en el entorno


In [7]:
# tu código
for coste in (-0.001, -0.04, -2.0):
    e = GridWorld(step_reward=coste)
    v, p, n = value_iteration(e, gamma=0.9)
    ev = evaluate(GridWorld(step_reward=coste), GreedyTabularPolicy(p), episodes=300, base_seed=0)
    print(f'coste {coste:>7} · {n:>2} barridos · {ev}')
    print(e.render_values(v, p), '\n')

coste  -0.001 · 41 barridos · avida          retorno +0.991 [+0.991, +0.992]  exito 100.0%  pasos   9.6
+0.62>  +0.71>  +0.82>  +0.94>   +1    
+0.54^    ###   +0.71^  +0.65<   -1    
+0.48^  +0.53>  +0.61^    ###   +0.35v 
+0.42^  +0.47^  +0.52^  +0.46<  +0.40<  

coste   -0.04 · 35 barridos · avida          retorno +0.636 [+0.603, +0.670]  exito 98.0%  pasos   9.1
+0.48>  +0.61>  +0.75>  +0.93>   +1    
+0.37^    ###   +0.61^  +0.59^   -1    
+0.28^  +0.36>  +0.47^    ###   +0.10v 
+0.21^  +0.27^  +0.35^  +0.26<  +0.17<  

coste    -2.0 · 29 barridos · avida          retorno -13.393 [-13.744, -13.043]  exito  6.0%  pasos   7.3
-6.54>  -4.47>  -2.31>  +0.31>   +1    
-8.18^    ###   -3.66>  -1.29>   -1    
-9.20>  -7.71>  -5.86^    ###   -1.46^ 
-10.11>  -8.85>  -7.44>  -5.90>  -3.94^  



**Explica.** ¿Qué está optimizando exactamente el agente para comportarse así?

_Tu respuesta:_ Con un coste por paso de −2, el agente reduce el número medio de pasos hasta 7.3, frente a 9.1 con un coste de −0.04. Sin embargo, esta reducción tiene un efecto negativo muy fuerte sobre el desempeño: la tasa de éxito cae de 98.0 % a solo 6.0 % y el retorno medio pasa a −13.393.

Quiere terminar asi sea de manera negativa en una trampa para asi no tener mas peso negativo ya que la reconpensa por no llegar y llegar en muy baja en comparacion del peso de moverse por eso prefiere terminar lo mas pronto posible. Esto me sorprendio porque no lo habia pensando. 

---
## Ejercicio 6 · El error plantado


In [ ]:
# ejecuta experiments/divergencia.py desde la terminal y pega aquí lo que salga
UPTC · Sesion 1 · Que sostiene la convergencia

  1) value_iteration(env, gamma=1.0)
     ValueError: gamma debe estar en [0, 1); se recibio 1.0
     La guardia protege una garantia: sin gamma < 1 el operador de
     Bellman deja de ser una contraccion.

  2) gamma = 1.0, recompensa por paso -0.04  (el entorno de siempre)
   barrido       V(3,0)       max|V|       cambio
  ------------------------------------------------
         1      -0.0400       0.7920     0.792000
         5      -0.2000       0.9513     0.337498
        10       0.4938       0.9685     0.158584
        50       0.6675       0.9721     0.000000
       100       0.6675       0.9721     0.000000
       500       0.6675       0.9721     0.000000
      1000       0.6675       0.9721     0.000000
      2000       0.6675       0.9721     0.000000
     Converge. Quedarse dando vueltas cuesta -0.04 por paso, o sea
     -infinito, asi que ninguna politica que el max prefiera lo hace.
     Es un camino mas corto estocastico, y ahi gamma = 1 esta bien
     definido. Perder la garantia no es perder la convergencia.

  3) gamma = 1.0, recompensa por paso +0.01  (ahora le pagamos por moverse)
   barrido       V(3,0)       max|V|       cambio
  ------------------------------------------------
         1       0.0100       0.8020     0.802000
         5       0.0500       0.9703     0.368426
        10       0.8791       1.0174     0.180948
        50       1.4173       1.4173     0.010000
       100       1.9173       1.9173     0.010000
       500       5.9173       5.9173     0.010000
      1000      10.9173      10.9173     0.010000
      2000      20.9173      20.9173     0.010000
     No converge. A partir del barrido 100 los valores crecen +0.01
     por barrido, indefinidamente, y el cambio se estanca en 0.01:
     el criterio de parada nunca se dispara.

  El mismo entorno con gamma = 0.9 converge en 42 barridos,
  con max|V| = 0.9496. La unica diferencia es el descuento.

  El diagnostico tiene dos capas.
    Matematica: con gamma = 1 existe una politica que nunca termina y
    acumula +0.01 sin fin, luego su retorno es +infinito. No hay punto
    fijo finito al que converger.
    De diseno: el error no fue poner gamma = 1, fue pagar por el
    proceso en vez de por el resultado. El descuento solo lo tapaba.

**Explica las dos capas del diagnóstico.**

_La capa matemática:_ 

_La capa de diseño:_ 


---
## Cierre

**Lo que más me sorprendió hoy:** lo que mas me sorprendio fue el ejercicio 5 en el que vemos que si el peso es muy grande tambien perdemos visibilidad porque no le importa llegar bien o mal solo le importa llegar

**Lo que todavía no entiendo:**  no entiendo del todo como 
